# Can we predict `arrears_next_month`?
Protocol: `protocol.yaml`. Temporal holdout from `holdout.csv`; 5-fold walk-forward CV on `billing_month` inside the training rows; selection by mean CV ROC AUC.

**Feature decision:** `final_notice_sent` is excluded. Per `columns.csv` it is recorded in the billing cycle *after* the billing month, i.e. around/after the outcome, so it is not known at prediction time (its 87% vs 1.4% arrears rate shows it is outcome leakage). `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('data.csv')
holdout = pd.read_csv('holdout.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
is_hold = df.index.isin(holdout['row'].to_numpy())
assert is_hold.sum()==7536 and len(df)==20939
train, hold = df[~is_hold], df[is_hold]
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min())

In [ ]:
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM   # final_notice_sent excluded: recorded after the outcome
def pre(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
# all candidates (with fixed hyperparameters) are named before any is scored
candidates = {
 'baseline_prior': Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':    Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':      Pipeline([('p',pre(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'hgb_depth3':     Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
 'hgb_depth6':     Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
# expanding-window walk-forward CV by month; the first ~1/6 of months is the minimum training window
months = np.sort(train.billing_month.unique())
blocks = np.array_split(months[len(months)//6:], 5)
cv_folds=[]
for b in blocks:
    tr = train.index[train.billing_month < b[0]]
    va = train.index[train.billing_month.isin(b)]
    cv_folds.append((tr,va))
    print(train.billing_month[tr].max(), '->', b[0], b[-1], len(tr), len(va))
cv_scores={}
for name,model in candidates.items():
    s=[]
    for tr,va in cv_folds:
        m=clone(model).fit(df.loc[tr,FEATURES], df.loc[tr,TARGET])   # preprocessing is fit inside each fold
        s.append(float(roc_auc_score(df.loc[va,TARGET], m.predict_proba(df.loc[va,FEATURES])[:,1])))
    cv_scores[name]=s
cv_table=pd.DataFrame(cv_scores).T; cv_table['mean']=cv_table.mean(axis=1); cv_table

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen by mean CV roc_auc:', chosen)
final = clone(candidates[chosen]).fit(train[FEATURES], train[TARGET])
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))  # scored once
print('holdout roc_auc:', round(holdout_auc,4))

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[[int(i) for i in tr], [int(i) for i in va]] for tr,va in cv_folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold.index],
 'holdout_scores': {chosen: {'roc_auc': holdout_auc}},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
The CV table and holdout ROC AUC above are the result. Without `final_notice_sent` (a post-outcome column), prediction relies on tariff, occupancy, usage, bill and late-payment history only. The holdout is a later period (2025-07 to 2025-12), so the score reflects forward-in-time generalisation.